In [1]:
from PIL import Image

import torch
import torch.nn as nn 
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torchvision.transforms.v2 import MixUp, CutMix, RandomChoice

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from timm.utils import ModelEmaV3
from timm.data import ImageNetInfo

from CNN import ImageNeuralNetwork

In [2]:
# Download directly to local Colab disk (fast, ~20-45 seconds)
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip -P /content/

# Unzip locally (fast, no Drive involved)
!unzip -q /content/tiny-imagenet-200.zip -d /content/

# Restructure val/ into class subfolders (train/ is already correctly structured)
import os
import shutil

val_dir = '/content/tiny-imagenet-200/val'
images_dir = os.path.join(val_dir, 'images')
annotations_file = os.path.join(val_dir, 'val_annotations.txt')

with open(annotations_file, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        filename, class_id = parts[0], parts[1]
        class_dir = os.path.join(val_dir, class_id)
        os.makedirs(class_dir, exist_ok=True)
        src = os.path.join(images_dir, filename)
        dst = os.path.join(class_dir, filename)
        if os.path.exists(src):
            shutil.move(src, dst)

shutil.rmtree(images_dir, ignore_errors=True)

URL transformed to HTTPS due to an HSTS policy
--2026-08-01 02:52:32--  https://cs231n.stanford.edu/tiny-imagenet-200.zip
Resolving cs231n.stanford.edu (cs231n.stanford.edu)... 171.64.64.64
Connecting to cs231n.stanford.edu (cs231n.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 248100043 (237M) [application/zip]
Saving to: ‘/content/tiny-imagenet-200.zip.1’

tiny-imagenet-200.z 100%[===================>] 236.61M  56.4MB/s    in 5.2s    

2026-08-01 02:52:37 (45.7 MB/s) - ‘/content/tiny-imagenet-200.zip.1’ saved [248100043/248100043]

replace /content/tiny-imagenet-200/words.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
best_accuracy = 70.12

def check_accuracy(cnn):
    global best_accuracy 
    correct = 0
    total = 0
    cnn.eval()

    with torch.no_grad(): 
        for data in test_loader:
            images, labels = data
            
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = cnn(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    cnn.train()
    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy}%')
    if(accuracy > best_accuracy):
        best_accuracy = accuracy
        torch.save(cnn.state_dict(), f'trained_net_{accuracy}.pth')
    
def load_image(image_path, new_transform):
    image = Image.open(image_path).convert('RGB')
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [5]:
train_transform = transforms.Compose([
    transforms.RandomCrop(64, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandAugment(num_ops=2, magnitude=9),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),  
    transforms.RandomErasing(p=0.25),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),
])

In [ ]:
train_data = torchvision.datasets.ImageFolder(root='data/tiny-imagenet-200/train', transform=train_transform)
test_data = torchvision.datasets.ImageFolder(root='data/tiny-imagenet-200/val', transform=test_transform
)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=1024, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1024, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)

In [7]:
num_epochs = 300
net = ImageNeuralNetwork(64, 4, 6, 200).to(device)
ema_net = ModelEmaV3(net, decay = 0.999)
loss_function = nn.CrossEntropyLoss(label_smoothing = 0.1)
optimizer = optim.SGD(net.parameters(), lr = 0.2, momentum = 0.9, weight_decay = 1e-4, nesterov = True)
scheduler = CosineAnnealingLR(optimizer, T_max = num_epochs, eta_min = 1e-7)

In [8]:
mixup = MixUp(alpha=0.2, num_classes=200)
cutmix = CutMix(alpha=1.0, num_classes=200)
mixup_cutmix = RandomChoice([mixup, cutmix])

In [ ]:
NUM_NO_MIX = 15
for epoch in range(1, num_epochs + 1):
    
    print(f'Training epoch {epoch}...\n')
    
    running_loss = 0.0
    correct = 0
    total = 0

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        if epoch <= num_epochs - NUM_NO_MIX:
            inputs, labels = mixup_cutmix(inputs, labels)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        ema_net.update(net)

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        if labels.dim() == 1:   # only count when NOT mixup/cutmix (hard labels)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    print(f'Loss: {running_loss / len(train_loader):.4f}, LR: {current_lr:.6f}')

    if total > 0:
        train_accuracy = 100 * correct / total
        print(f'Train Accuracy: {train_accuracy:.2f}%\n')
    
    if(epoch >= 140 and epoch % 20 == 0):
        check_accuracy(ema_net.module)

Training epoch 1...

Loss: 5.2284, LR: 0.199995
Training epoch 2...

Loss: 4.9708, LR: 0.199978
Training epoch 3...

Loss: 4.7952, LR: 0.199951
Training epoch 4...

Loss: 4.5908, LR: 0.199912
Training epoch 5...

Loss: 4.4976, LR: 0.199863
Training epoch 6...

Loss: 4.4671, LR: 0.199803
Training epoch 7...

Loss: 4.3488, LR: 0.199731
Training epoch 8...

Loss: 4.2051, LR: 0.199649
Training epoch 9...

Loss: 4.1952, LR: 0.199556
Training epoch 10...

Loss: 4.1193, LR: 0.199452
Training epoch 11...

Loss: 4.0924, LR: 0.199337
Training epoch 12...

Loss: 4.0324, LR: 0.199211
Training epoch 13...

Loss: 4.1128, LR: 0.199075
Training epoch 14...

Loss: 3.9856, LR: 0.198927
Training epoch 15...

Loss: 3.9189, LR: 0.198769
Training epoch 16...

Loss: 3.8652, LR: 0.198600
Training epoch 17...

Loss: 3.9098, LR: 0.198420
Training epoch 18...

Loss: 3.8767, LR: 0.198229
Training epoch 19...

Loss: 3.7969, LR: 0.198027
Training epoch 20...

Loss: 3.7542, LR: 0.197815
Training epoch 21...

Loss: 3

In [ ]:
net = ImageNeuralNetwork(64, 4, 6, 200).to(device)
net.load_state_dict(torch.load(f'trained_net_{best_accuracy}.pth', map_location = device))

In [ ]:
new_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),
])

In [ ]:
image_paths = ['/home/simon/Tiny-ImageNet-Image-Classifier/python-backend/uni.jpg']
images = [load_image(img, new_transform) for img in image_paths]
image_info = ImageNetInfo(subset='imagenet-1k')
net.eval()
with torch.no_grad():
    for image in images:
        outputs = net(image.to(device))
        _, predicted = torch.max(outputs, 1)
        wnid = train_data.classes[predicted.item()]
        print(f'Prediction: {image_info.label_name_to_description(wnid)}')